# 第 52 课：MCP Streamable HTTP、远端身份与授权边界

**网络连通、身份凭证有效、拥有某项权限、获得本次执行批准，是不同结论。**

本课把第 51 课的工具服务从 stdio 迁移到 HTTP：使用固定 `2026-07-28` 协议，实际运行 JSON 与请求内 SSE 两条响应路径，并验证身份、scope、对象归属和宿主白名单。

**材料可以独立运行，不需要前几课源码。** Python 3.10+，只有标准库；不下载模型，不调用外部 API，不访问真正的授权服务器。服务和客户端仅使用 `127.0.0.1`。

> 本课不是完整 OAuth 实现。开发凭证是宿主预先创建的随机 opaque 字符串，服务端从自己的可信记录查到其 subject、issuer、audience、scope 和到期时间。不使用未验签 JWT 中的字段。没有浏览器登录、授权码、PKCE、refresh、TLS 或生产安全认证。

## 学习目标与阅读方法

完成后，应能解释同一个合法 HTTP 请求为什么可能返回 401、403、400、RPC error，或 `200 + isError=true`；能为一次工具调用同时构造请求正文和镜像头；能验证 SSE 最终结果而不是把 EOF 当作成功。

本课完整实现较长。**定义单元已设置为可折叠，先运行，再按下面的实验阅读指定函数。** 重点阅读 `validate_grant()`、`require_scope()`、`validate_mirrors()`、`HttpClient.request()`；HTTP 协议处理和检查函数可作为工程参考。

所有服务实验用 `with LocalService(...)` 退出时关闭。不要在 Notebook 中直接运行长期阻塞的 `serve()`。

In [1]:
from __future__ import annotations
import sys
print("Python:", sys.version.split()[0])
print("默认实验：标准库 + 真实本地 HTTP；不使用模型和 GPU。")

Python: 3.13.5
默认实验：标准库 + 真实本地 HTTP；不使用模型和 GPU。


## 1．先锁定协议版本，再阅读教程

本课固定 `2026-07-28`，延续第 51 课。该修订版使用逐请求元数据，不需要旧式 `initialize` 握手，也没有旧的 `Mcp-Session-Id` 会话和独立 GET 监听流。

| 操作 | 本课行为 |
|---|---|
| `POST /mcp` | 一个 JSON-RPC 请求，返回单个 JSON 或本请求的 SSE |
| `GET /mcp`、`DELETE /mcp` | 405 |
| `GET /.well-known/oauth-protected-resource/mcp` | 公开的受保护资源元数据 |
| `server/discover` | 返回协议版本与能力，**不是 OAuth 授权发现** |

SSE 中可以先出现关联本次请求的进度通知，随后返回最终 JSON-RPC 响应。这里没有第 49 课自定义的 `ready/delta/done` 事件，也没有可恢复事件日志。来源：[1][2]。

### 基础定义
先运行本单元。`strict_loads()` 拒绝重复键与非有限数值；`endpoint_parts()` 将实际网络访问限制在明确的回环地址，避免把本课客户端当成任意 URL 抓取器。

In [2]:
import argparse
import base64
import codecs
import copy
import hashlib
import http.client
import importlib.metadata
import json
import math
import os
import platform
import re
import secrets
import socket
import sys
import tempfile
import threading
import time
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, Inexact, Rounded, localcontext
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any, Callable
from urllib.parse import urlsplit

PROTOCOL = "2026-07-28"
VERSION = "lesson52-http-auth-v1"
META = "io.modelcontextprotocol/"
ISSUER = "https://auth.lesson52.example"  # 保留示例域名；本课绝不访问它。
METADATA_PATH = "/.well-known/oauth-protected-resource/mcp"
MAX_BODY = 32768
MAX_RESPONSE = 131072
MAX_EVENTS = 32
SERVER_INFO = {"name": "lesson52-tools", "version": "1.0.0"}
SOURCES = {
    "transport": "https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/streamable-http",
    "authorization": "https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization",
    "discovery": "https://modelcontextprotocol.io/specification/2026-07-28/server/discover",
    "tools": "https://modelcontextprotocol.io/specification/2026-07-28/server/tools",
    "security": "https://modelcontextprotocol.io/docs/2026-07-28/tutorials/security/security_best_practices",
    "oauth_metadata": "https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/authorization-server-discovery",
    "sdk_authorization": "https://py.sdk.modelcontextprotocol.io/run/authorization/",
    "sse": "https://html.spec.whatwg.org/multipage/server-sent-events.html",
    "http_server": "https://docs.python.org/3/library/http.server.html",
}

class ContractError(ValueError):
    pass

class TransportError(RuntimeError):
    pass

class HTTPFailure(RuntimeError):
    def __init__(self, status: int, code: str, headers: dict | None = None):
        super().__init__(f"HTTP {status}: {code}")
        self.status, self.code, self.headers = status, code, headers or {}

class RPCFailure(RuntimeError):
    def __init__(self, code: int, message: str, status: int = 200):
        super().__init__(f"RPC {code}: {message}")
        self.code, self.status = code, status

class GrantRejected(ValueError):
    pass


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def strict_loads(text: str) -> Any:
    def pairs(items):
        out = {}
        for k, v in items:
            if k in out:
                raise ContractError("重复 JSON 键")
            out[k] = v
        return out

    def constant(_):
        raise ContractError("非标准 JSON 数值")

    try:
        result = json.loads(text, object_pairs_hook=pairs, parse_constant=constant)
    except (json.JSONDecodeError, RecursionError, UnicodeError) as exc:
        raise ContractError("JSON 解析失败") from exc
    count = 0

    def walk(node, depth=0):
        nonlocal count
        count += 1
        if depth > 20 or count > 6000:
            raise ContractError("JSON 超出结构限制")
        if isinstance(node, float) and not math.isfinite(node):
            raise ContractError("非有限数值")
        if isinstance(node, dict):
            for v in node.values():
                walk(v, depth + 1)
        elif isinstance(node, list):
            for v in node:
                walk(v, depth + 1)
        elif isinstance(node, str):
            try:
                node.encode("utf-8")
            except UnicodeError as exc:
                raise ContractError("无效 Unicode") from exc
    walk(result)
    return result


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ContractError(message)


def endpoint_parts(endpoint: str):
    """教学客户端不访问任意网络地址；TLS/公网远端需真实基础设施。"""
    require(isinstance(endpoint, str), "endpoint 必须为字符串")
    try:
        p = urlsplit(endpoint)
        port = p.port
    except ValueError as exc:
        raise ContractError("endpoint 无效") from exc
    require(p.scheme == "http" and p.hostname == "127.0.0.1", "仅允许本机 127.0.0.1 HTTP")
    require(p.username is None and p.password is None and not p.query and not p.fragment,
            "不接受 URL 凭证、查询串或片段")
    require(p.path == "/mcp" and port is not None and 1 <= port <= 65535, "固定 /mcp 和显式端口")
    require(endpoint == f"http://127.0.0.1:{port}/mcp", "请使用本课规范 endpoint")
    return p

In [3]:
for text in ['{"value":16}', '{"value":1,"value":2}', '{"value":NaN}']:
    try:
        print("解析：", strict_loads(text))
    except ContractError as error:
        print("拒绝：", str(error))

解析： {'value': 16}
拒绝： 重复 JSON 键
拒绝： 非标准 JSON 数值


## 2．HTTP 镜像头：路由信息与正文必须相同

每个请求包含 `MCP-Protocol-Version`、`Mcp-Method`；`tools/call` 还要带 `Mcp-Name`。它们镜像正文中的版本、方法和工具名，不能由两份互不核对的数据独立决定。

例如，网关看到 `Mcp-Name: read_doc` 而实际正文要求 `calculate`，可能导致权限判断与实际执行不一致。规范要求服务拒绝镜像不匹配，返回 HTTP 400 和 `-32020 HeaderMismatch`。来源：[1]。

本课工具名是 ASCII；辅助函数也演示了规范的 Base64 sentinel 编码。未实现自定义 `x-mcp-header`、resources、prompts 等扩展，不声称是完整客户端。

### 请求构造与响应校验
`make_request()` 复制输入再添加 `_meta`；`request_headers()` 从同一个正文生成头；`validate_mirrors()` 在执行前核对。`resultType=complete` 只表示结果完整，不表示业务成功。

In [4]:
def encode_header(value: str) -> str:
    require(isinstance(value, str), "镜像头来源必须为字符串")
    plain = (value == value.strip() and all(32 <= ord(c) <= 126 for c in value)
             and not (value.startswith("=?base64?") and value.endswith("?=")))
    if plain:
        return value
    return "=?base64?" + base64.b64encode(value.encode("utf-8")).decode("ascii") + "?="


def decode_header(value: str) -> str:
    require(isinstance(value, str) and len(value) <= 4096, "头值不合法")
    if value.startswith("=?base64?") and value.endswith("?="):
        try:
            return base64.b64decode(value[9:-2], validate=True).decode("utf-8")
        except (ValueError, UnicodeError) as exc:
            raise ContractError("Base64 镜像头错误") from exc
    require(value == value.strip() and all(32 <= ord(c) <= 126 for c in value), "不安全的明文头值")
    return value


def make_request(method: str, params: dict | None = None, request_id: int | str = 1,
                 client_name: str = "lesson52-client", progress_token: str | None = None) -> dict:
    require(isinstance(method, str) and re.fullmatch(r"[a-zA-Z_/]+", method) is not None, "method 不合法")
    require((type(request_id) is int and 0 <= request_id < 2**53)
            or (isinstance(request_id, str) and 0 < len(request_id) <= 64), "id 不合法")
    p = copy.deepcopy({} if params is None else params)
    require(isinstance(p, dict) and "_meta" not in p, "params 必须是字典，元数据由客户端添加")
    p["_meta"] = {META + "protocolVersion": PROTOCOL,
                  META + "clientInfo": {"name": client_name, "version": "1.0.0"},
                  META + "clientCapabilities": {}}
    if progress_token is not None:
        require(isinstance(progress_token, str) and 0 < len(progress_token) <= 64, "progress token 不合法")
        p["_meta"]["progressToken"] = progress_token
    return {"jsonrpc": "2.0", "id": request_id, "method": method, "params": p}


def request_headers(body: dict) -> dict[str, str]:
    out = {"Content-Type": "application/json",
           "Accept": "application/json, text/event-stream",
           "MCP-Protocol-Version": body["params"]["_meta"][META + "protocolVersion"],
           "Mcp-Method": body["method"]}
    if body["method"] == "tools/call":
        out["Mcp-Name"] = encode_header(body["params"]["name"])
    return out


def validate_mirrors(body: dict, headers: dict[str, str]) -> None:
    """本课只支持 ASCII 工具名，无自定义 x-mcp-header schema。"""
    p = body.get("params")
    require(isinstance(p, dict) and isinstance(p.get("_meta"), dict), "缺少请求 _meta")
    m = p["_meta"]
    require(isinstance(m.get(META + "clientCapabilities"), dict), "缺少客户端能力")
    require(headers.get("mcp-protocol-version") == m.get(META + "protocolVersion")
            and isinstance(m.get(META + "protocolVersion"), str), "协议版本镜像不一致")
    require(headers.get("mcp-method") == body.get("method"), "方法镜像不一致")
    if body.get("method") == "tools/call":
        require(isinstance(p.get("name"), str), "缺少工具名")
        require(decode_header(headers.get("mcp-name")) == p["name"], "工具名镜像不一致")
    elif "mcp-name" in headers:
        raise ContractError("本课无名称的方法不能附加 Mcp-Name")


def rpc_result(request_id, fields: dict) -> dict:
    return {"jsonrpc": "2.0", "id": request_id,
            "result": {"resultType": "complete", **fields,
                       "_meta": {META + "serverInfo": copy.deepcopy(SERVER_INFO)}}}


def rpc_error(request_id, code: int, message: str, data=None) -> dict:
    error = {"code": code, "message": message}
    if data is not None:
        error["data"] = data
    return {"jsonrpc": "2.0", "id": request_id, "error": error}


def validate_reply(reply: dict, expected_id) -> dict:
    require(isinstance(reply, dict) and reply.get("jsonrpc") == "2.0", "不是 RPC 响应")
    require(type(reply.get("id")) is type(expected_id) and reply["id"] == expected_id,
            "RPC id 不匹配")
    require(("result" in reply) != ("error" in reply), "必须恰有 result 或 error")
    if "error" in reply:
        e = reply["error"]
        require(isinstance(e, dict) and type(e.get("code")) is int and isinstance(e.get("message"), str), "非法 RPC error")
        raise RPCFailure(e["code"], e["message"])
    result = reply["result"]
    require(isinstance(result, dict) and result.get("resultType") == "complete", "本课不处理 MRTR/异步任务")
    return result

In [5]:
sample_request = make_request(
    "tools/call",
    {"name": "calculate", "arguments": {"op": "subtract", "left": "30", "right": "14"}},
    request_id=7,
)
print(json.dumps(sample_request, ensure_ascii=False, indent=2))
print(json.dumps(request_headers(sample_request), ensure_ascii=False, indent=2))
print("凭证没有放在 JSON 中；真实客户端另加 Authorization 头。")

{
  "jsonrpc": "2.0",
  "id": 7,
  "method": "tools/call",
  "params": {
    "name": "calculate",
    "arguments": {
      "op": "subtract",
      "left": "30",
      "right": "14"
    },
    "_meta": {
      "io.modelcontextprotocol/protocolVersion": "2026-07-28",
      "io.modelcontextprotocol/clientInfo": {
        "name": "lesson52-client",
        "version": "1.0.0"
      },
      "io.modelcontextprotocol/clientCapabilities": {}
    }
  }
}
{
  "Content-Type": "application/json",
  "Accept": "application/json, text/event-stream",
  "MCP-Protocol-Version": "2026-07-28",
  "Mcp-Method": "tools/call",
  "Mcp-Name": "calculate"
}
凭证没有放在 JSON 中；真实客户端另加 Authorization 头。


In [6]:
sample_headers = {k.lower(): v for k, v in request_headers(sample_request).items()}
validate_mirrors(sample_request, sample_headers)
print("正确镜像：通过")
try:
    validate_mirrors(sample_request, {**sample_headers, "mcp-name": "read_doc"})
except ContractError as error:
    print("冲突镜像：", error)
print("中文名称头的往返：", decode_header(encode_header("资料读取")))

正确镜像：通过
冲突镜像： 工具名镜像不一致
中文名称头的往返： 资料读取


## 3．凭证到底证明了什么？

生产 HTTP 授权通常分为三方：授权服务器负责用户交互及签发访问凭证；MCP 服务是资源服务器，验证每个请求的凭证；客户端获得适用于目标资源的凭证后发送调用。来源：[3][6]。

| 字段 | 问题 |
|---|---|
| `issuer` | 来自哪个被信任的发行方？ |
| `audience` / resource | 这份凭证是发给哪个目标服务的？ |
| `expires_at` | 现在是否仍有效？ |
| `subject` | 本服务将这个已验证凭证关联到哪个主体？ |
| `scopes` | 此凭证允许哪些操作类别？ |

**凭证持有者不等于已经现场认证过的真人。** `clientInfo.name="administrator"`、模型自报身份和 `serverInfo.name` 都不是认证依据。服务名称由服务器自报，只适合展示和调试。来源：[2]。

本课用服务端预配记录替代真正的签发与 token 验证基础设施，目的只是观察校验边界。不能把它包装成 OAuth 登录或 JWT 验签。

### 身份与两个有限工具
`DevelopmentTokenStore` 只保存随机凭证的 SHA256 摘要及可信记录，没有网络签发或刷新接口。`read_doc` 需要 `docs:read`；`calculate` 需要 `calc:use`。两者都是只读教学工具。

In [7]:
@dataclass(frozen=True)
class Principal:
    subject: str
    scopes: frozenset[str]
    issuer: str
    audience: str


def validate_grant(record: dict, resource: str, issuer: str, now: float) -> Principal:
    """只接收服务端可信记录。不是解码未验签 JWT，也不是 OAuth grant 流程。"""
    if record.get("issuer") != issuer:
        raise GrantRejected("issuer")
    if record.get("audience") != resource:
        raise GrantRejected("audience")
    if record.get("revoked") is not False:
        raise GrantRejected("revoked")
    if not (isinstance(record.get("expires_at"), (float, int)) and math.isfinite(record["expires_at"])
            and isinstance(record.get("not_before"), (float, int)) and math.isfinite(record["not_before"])
            and record["not_before"] <= now < record["expires_at"]):
        raise GrantRejected("time")
    if not isinstance(record.get("subject"), str) or not record["subject"]:
        raise GrantRejected("subject")
    scopes = record.get("scopes")
    if not isinstance(scopes, list) or any(s not in {"docs:read", "calc:use"} for s in scopes):
        raise GrantRejected("scopes")
    return Principal(record["subject"], frozenset(scopes), issuer, resource)


class DevelopmentTokenStore:
    """进程内开发夹具；只存 token 摘要，不提供网络签发/刷新接口。"""
    def __init__(self, resource: str, clock: Callable[[], float] = time.time):
        self.resource, self.clock = resource, clock
        self._records = {}
        self._lock = threading.Lock()

    def provision(self, subject: str, scopes: list[str], ttl: float = 900,
                  issuer: str = ISSUER, audience: str | None = None, not_before: float | None = None) -> str:
        token = "l52_" + secrets.token_urlsafe(32)
        record = {"subject": subject, "scopes": list(scopes), "issuer": issuer,
                  "audience": audience or self.resource, "expires_at": self.clock() + ttl,
                  "not_before": self.clock() if not_before is None else not_before, "revoked": False}
        with self._lock:
            self._records[hashlib.sha256(token.encode()).hexdigest()] = record
        return token

    def revoke(self, token: str) -> None:
        with self._lock:
            record = self._records.get(hashlib.sha256(token.encode()).hexdigest())
            if record:
                record["revoked"] = True

    def verify(self, authorization: str | None) -> Principal:
        if not isinstance(authorization, str) or not authorization.startswith("Bearer "):
            raise GrantRejected("missing")
        token = authorization[7:]
        if not re.fullmatch(r"l52_[A-Za-z0-9_-]{40,80}", token):
            raise GrantRejected("unknown")
        key = hashlib.sha256(token.encode()).hexdigest()
        with self._lock:
            record = copy.deepcopy(self._records.get(key))
        if record is None:
            raise GrantRejected("unknown")
        return validate_grant(record, self.resource, ISSUER, self.clock())


TOOL_SCOPES = {"read_doc": "docs:read", "calculate": "calc:use"}
DOCS = {
    "backup": {"owner": None, "text": "自动备份保留 14 天。虚构的教学资料。"},
    "trash": {"owner": None, "text": "回收站文件保留 30 天。虚构的教学资料。"},
    "alice_private": {"owner": "alice", "text": "Alice 的教学笔记：先检查资料，再进行计算。"},
    "bob_private": {"owner": "bob", "text": "Bob 的教学笔记：权限范围不等于全部对象权限。"},
}


def require_scope(principal: Principal, name: str) -> None:
    if TOOL_SCOPES[name] not in principal.scopes:
        raise PermissionError("insufficient_scope")


def require_document_access(principal: Principal, doc_id: str) -> None:
    doc = DOCS.get(doc_id)
    if doc is not None and doc["owner"] not in (None, principal.subject):
        raise PermissionError("object_forbidden")


def object_schema(properties: dict) -> dict:
    return {"type": "object", "properties": properties,
            "required": list(properties), "additionalProperties": False}


TOOL_SCHEMAS = {
    "calculate": {"name": "calculate", "description": "有界十进制计算，不处理表达式或单位。",
        "inputSchema": object_schema({"op": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
                                      "left": {"type": "string"}, "right": {"type": "string"}}),
        "annotations": {"readOnlyHint": True, "destructiveHint": False, "idempotentHint": True}},
    "read_doc": {"name": "read_doc", "description": "读取登记的虚构资料；先检查 scope，再检查归属。",
        "inputSchema": object_schema({"doc_id": {"type": "string"}}),
        "annotations": {"readOnlyHint": True, "destructiveHint": False, "idempotentHint": True}},
}
for _schema in TOOL_SCHEMAS.values():
    _schema["outputSchema"] = object_schema({"ok": {"type": "boolean"},
        "data": {"type": ["object", "null"]}, "error": {"type": ["string", "null"]}})


def validate_arguments(name: str, args: Any) -> dict:
    require(isinstance(args, dict), "arguments 必须是对象")
    if name == "read_doc":
        require(set(args) == {"doc_id"} and isinstance(args["doc_id"], str)
                and re.fullmatch(r"[a-z][a-z0-9_]{0,31}", args["doc_id"]) is not None, "doc_id 不合法")
    else:
        require(set(args) == {"op", "left", "right"}, "计算参数字段不匹配")
        require(args["op"] in {"add", "subtract", "multiply", "divide"}, "运算不支持")
        for key in ("left", "right"):
            require(isinstance(args[key], str) and re.fullmatch(r"-?(?:0|[1-9][0-9]{0,11})(?:\.[0-9]{1,6})?", args[key]) is not None,
                    "数值必须是有界十进制字符串，不执行表达式")
    return copy.deepcopy(args)


def tool_result(data: dict | None = None, error: str | None = None) -> dict:
    record = {"ok": error is None, "data": data if error is None else None, "error": error}
    return {"isError": error is not None, "content": [{"type": "text", "text": canonical(record)}],
            "structuredContent": record}


def validate_tool_result(result: dict) -> dict:
    require(type(result.get("isError")) is bool, "缺少 isError")
    data = result.get("structuredContent")
    require(isinstance(data, dict) and set(data) == {"ok", "data", "error"}
            and type(data["ok"]) is bool, "structuredContent 不符合本课契约")
    require(data["ok"] == (not result["isError"]), "结果状态冲突")
    if data["ok"]:
        require(isinstance(data["data"], dict) and data["error"] is None, "成功结果格式错误")
    else:
        require(data["data"] is None and isinstance(data["error"], str), "失败结果格式错误")
    c = result.get("content")
    require(isinstance(c, list) and len(c) == 1 and c[0].get("type") == "text"
            and strict_loads(c[0]["text"]) == data, "本课两种结果表示不一致")
    return data

In [8]:
trusted_record = {
    "subject": "alice", "scopes": ["docs:read"], "issuer": ISSUER,
    "audience": "teaching-resource", "not_before": 0,
    "expires_at": 100, "revoked": False,
}
for now in (99, 100):
    try:
        identity = validate_grant(trusted_record, "teaching-resource", ISSUER, now)
        print(now, "有效：", identity.subject, sorted(identity.scopes))
    except GrantRejected as error:
        print(now, "拒绝：", str(error))
print("到期边界 now == expires_at 时已经无效。")

99 有效： alice ['docs:read']
100 拒绝： time
到期边界 now == expires_at 时已经无效。


### Scope 仍然不是全部对象的通行证

拥有 `docs:read` 只说明可以尝试读资料；读取 `alice_private` 时，还要检查对象归属。反过来，宿主也可以在服务授权以外，设置更窄的模型工具白名单。

执行许可是这些限制的交集，而不是只要其中一个通过就能运行。工具注解 `readOnlyHint` 只是描述，不可代替服务的 scope 和对象检查。来源：[4][5]。

In [9]:
identity = validate_grant(trusted_record, "teaching-resource", ISSUER, 1)
require_scope(identity, "read_doc")
require_document_access(identity, "alice_private")
print("alice + docs:read → 自己的资料：允许")
for label, operation in [
    ("缺少 calc:use", lambda: require_scope(identity, "calculate")),
    ("读取 bob 的私有对象", lambda: require_document_access(identity, "bob_private")),
]:
    try:
        operation()
    except PermissionError as error:
        print(label, "→ 拒绝：", str(error))

alice + docs:read → 自己的资料：允许
缺少 calc:use → 拒绝： insufficient_scope
读取 bob 的私有对象 → 拒绝： object_forbidden


## 4．从入口到工具的检查顺序

```text
固定 Host / Origin / 路径 / 大小与媒体类型
    → 每个请求验证凭证
    → 严格 JSON 与镜像头
    → 受支持的版本和方法
    → 工具 scope
    → 参数与对象归属
    → 实际进入工具
```

`Origin` 反映浏览器请求上下文，CLI 可以不发送，也能自行设置；它不是用户身份。CORS 是浏览器机制，也不是授权系统。本课不实现浏览器 CORS。

服务只绑定 `127.0.0.1`，校验 Host 和存在时的 Origin。正式远端部署还需要正确的 HTTPS/TLS、可信授权服务器和完整客户端；本课不会把本机 HTTP 当作安全公网部署。来源：[1][5]。

### 资源服务的应用层
重点看 `Application.handle()`：身份校验不读取模型正文；`require_scope()` 和对象归属检查发生在执行计数增加之前。目录会因权限不同而变化，但调用端点还会独立执行同样的权限检查。

In [10]:
@dataclass
class Reply:
    status: int
    headers: dict
    body: dict
    stream: bool = False
    progress_token: str | None = None


class Application:
    def __init__(self, resource: str, response_mode: str = "json", clock: Callable[[], float] = time.time):
        endpoint_parts(resource)
        require(response_mode in {"json", "sse"}, "响应模式错误")
        self.resource = resource
        self.origin = resource[:-4]
        self.authority = urlsplit(resource).netloc
        self.mode = response_mode
        self.tokens = DevelopmentTokenStore(resource, clock)
        self._lock = threading.Lock()
        self.executions = 0
        self.audit = []

    def record(self, outcome: str, subject: str | None = None, method: str | None = None,
               tool: str | None = None) -> None:
        # 不记录原始 URL、凭证、任意正文或 clientInfo；不把 _meta 当身份。
        with self._lock:
            self.audit.append({"sequence": len(self.audit) + 1, "outcome": outcome,
                               "subject": subject, "method": method if method in {"server/discover", "tools/list", "tools/call"} else None,
                               "tool": tool if tool in TOOL_SCOPES else None})

    def metadata(self) -> dict:
        return {"resource": self.resource, "authorization_servers": [ISSUER],
                "scopes_supported": ["docs:read", "calc:use"],
                "bearer_methods_supported": ["header"],
                "lesson52_mode": "preprovisioned_opaque_development_credentials_no_oauth_login"}

    def challenge(self, status: int, reason: str, scope: str | None = None) -> Reply:
        parts = [f'Bearer resource_metadata="{self.origin}{METADATA_PATH}"']
        if reason in {"invalid_token", "insufficient_scope"}:
            parts.append(f'error="{reason}"')
        if scope:
            parts.append(f'scope="{scope}"')
        return Reply(status, {"WWW-Authenticate": ", ".join(parts)}, {"error": reason})

    def boundary(self, headers: dict, path: str) -> Reply | None:
        if headers.get("host") != self.authority:
            return Reply(403, {}, {"error": "invalid_host"})
        if "origin" in headers and headers["origin"] != self.origin:
            return Reply(403, {}, {"error": "invalid_origin"})
        if "?" in path or "#" in path:
            return Reply(400, {}, {"error": "query_or_fragment_not_allowed"})
        return None

    def handle(self, headers: dict, raw: bytes) -> Reply:
        try:
            principal = self.tokens.verify(headers.get("authorization"))
        except GrantRejected as exc:
            self.record("unauthenticated")
            return self.challenge(401, "authorization_required" if str(exc) == "missing" else "invalid_token")
        try:
            body = strict_loads(raw.decode("utf-8"))
            require(isinstance(body, dict) and set(body) == {"jsonrpc", "id", "method", "params"}
                    and body.get("jsonrpc") == "2.0", "本课只接受单个 JSON-RPC 请求")
            rid = body["id"]
            require((type(rid) is int and 0 <= rid < 2**53) or (isinstance(rid, str) and 0 < len(rid) <= 64), "id 错误")
            require(isinstance(body["method"], str), "method 错误")
        except (ContractError, UnicodeError):
            self.record("malformed_rpc", principal.subject)
            return Reply(400, {}, rpc_error(None, -32600, "Invalid request"))
        try:
            validate_mirrors(body, headers)
        except ContractError:
            self.record("header_mismatch", principal.subject)
            return Reply(400, {}, rpc_error(rid, -32020, "HeaderMismatch"))
        method, p = body["method"], body["params"]
        if p["_meta"][META + "protocolVersion"] != PROTOCOL:
            return Reply(400, {}, rpc_error(rid, -32022, "Unsupported protocol version", {"supported": [PROTOCOL]}))
        # 旧 Mcp-Session-Id/Last-Event-ID 不参与身份、路由或状态，按版本约定忽略。
        if method == "server/discover":
            if set(p) != {"_meta"}:
                return Reply(200, {}, rpc_error(rid, -32602, "Invalid params"))
            fields = {"supportedVersions": [PROTOCOL], "capabilities": {"tools": {}},
                      "ttlMs": 0, "cacheScope": "private"}
        elif method == "tools/list":
            if set(p) != {"_meta"}:
                return Reply(200, {}, rpc_error(rid, -32602, "No pagination needed for this two-tool lesson"))
            fields = {"tools": [copy.deepcopy(TOOL_SCHEMAS[n]) for n in sorted(TOOL_SCHEMAS)
                                if TOOL_SCOPES[n] in principal.scopes], "ttlMs": 0, "cacheScope": "private"}
        elif method == "tools/call":
            if set(p) != {"_meta", "name", "arguments"} or p.get("name") not in TOOL_SCOPES:
                return Reply(200, {}, rpc_error(rid, -32602, "Unknown tool or invalid call envelope"))
            name = p["name"]
            try:
                require_scope(principal, name)
            except PermissionError:
                self.record("scope_denied", principal.subject, method, name)
                return self.challenge(403, "insufficient_scope", TOOL_SCOPES[name])
            try:
                args = validate_arguments(name, p["arguments"])
            except (ContractError, TypeError):
                fields = tool_result(error="invalid_arguments")
            else:
                if name == "read_doc":
                    try:
                        require_document_access(principal, args["doc_id"])
                    except PermissionError:
                        self.record("object_denied", principal.subject, method, name)
                        return Reply(403, {}, {"error": "object_forbidden"})
                with self._lock:
                    self.executions += 1
                if name == "read_doc":
                    doc = DOCS.get(args["doc_id"])
                    fields = (tool_result({"doc_id": args["doc_id"], "text": doc["text"],
                                           "revision": digest(doc["text"])}) if doc else tool_result(error="not_found"))
                else:
                    a, b = Decimal(args["left"]), Decimal(args["right"])
                    if args["op"] == "divide" and b == 0:
                        fields = tool_result(error="division_by_zero")
                    else:
                        with localcontext() as ctx:
                            ctx.prec = 28
                            value = {"add": lambda: a+b, "subtract": lambda: a-b,
                                     "multiply": lambda: a*b, "divide": lambda: a/b}[args["op"]]()
                            rounded = bool(ctx.flags[Inexact] or ctx.flags[Rounded])
                        fields = tool_result({"value": str(value), "rounded": rounded, "unit_checked": False})
                self.record("tool_returned", principal.subject, method, name)
            progress = p["_meta"].get("progressToken")
            return Reply(200, {}, rpc_result(rid, fields), self.mode == "sse", progress)
        else:
            return Reply(404, {}, rpc_error(rid, -32601, "Method not found"))
        self.record("rpc_complete", principal.subject, method)
        return Reply(200, {}, rpc_result(rid, fields))

### 真正的 HTTP 传输层
运行这些定义不会启动长期服务。接下来每个实验通过上下文管理器启动、调用和关闭服务。

本课每个 HTTP 请求使用独立连接也能工作，没有全局“已经登录所以免验 token”的状态。开发凭证仍存放在本进程中；无协议会话不等于应用程序完全没有状态。

In [11]:
class LessonHTTPServer(ThreadingHTTPServer):
    daemon_threads = True
    allow_reuse_address = True


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    server_version = "Lesson52/1"

    def setup(self):
        super().setup()
        self.connection.settimeout(3.0)

    def log_message(self, *_):
        pass  # 不将 Authorization、URL 查询串或请求正文写进默认日志。

    @property
    def app(self) -> Application:
        return self.server.app

    def unique_headers(self):
        protected = {"authorization", "host", "origin", "content-length", "content-type", "accept",
                     "mcp-method", "mcp-name", "mcp-protocol-version"}
        out = {}
        for name in self.headers.keys():
            key = name.lower()
            if key in protected and len(self.headers.get_all(name)) != 1:
                raise ContractError("duplicate_header")
            out[key] = self.headers[name]
        return out

    def send_reply(self, reply: Reply):
        self.close_connection = True
        headers = {"Cache-Control": "no-store", "Connection": "close", **reply.headers}
        try:
            if reply.stream:
                self.send_response(reply.status)
                self.send_header("Content-Type", "text/event-stream")
                self.send_header("X-Accel-Buffering", "no")
                for k, v in headers.items():
                    self.send_header(k, v)
                self.end_headers()
                self.wfile.write(b": lesson52 request-scoped stream\n\n")
                self.wfile.flush()
                if reply.progress_token is not None:
                    progress = {"jsonrpc": "2.0", "method": "notifications/progress",
                                "params": {"progressToken": reply.progress_token, "progress": 1, "total": 1}}
                    self.wfile.write(sse_frame(progress))
                    self.wfile.flush()
                self.wfile.write(sse_frame(reply.body))
                self.wfile.flush()
            else:
                data = canonical(reply.body).encode("utf-8")
                self.send_response(reply.status)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(data)))
                for k, v in headers.items():
                    self.send_header(k, v)
                self.end_headers()
                self.wfile.write(data)
        except (BrokenPipeError, ConnectionResetError, socket.timeout):
            # 只读工具可能已经完成；发送失败不会导致自动重做。
            self.app.record("response_transport_lost")

    def preflight(self):
        try:
            headers = self.unique_headers()
        except ContractError:
            self.send_reply(Reply(400, {}, {"error": "duplicate_header"}))
            return None
        denied = self.app.boundary(headers, self.path)
        if denied:
            self.send_reply(denied)
            return None
        return headers

    def do_GET(self):
        headers = self.preflight()
        if headers is None:
            return
        if self.path == METADATA_PATH:
            self.app.record("metadata_with_authorization" if "authorization" in headers else "metadata_without_authorization")
            self.send_reply(Reply(200, {}, self.app.metadata()))
        elif self.path == "/healthz":
            self.send_reply(Reply(200, {}, {"ready": True, "protocol": PROTOCOL, "local_development": True}))
        elif self.path == "/mcp":
            self.send_reply(Reply(405, {"Allow": "POST"}, {"error": "method_not_allowed"}))
        else:
            self.send_reply(Reply(404, {}, {"error": "not_found"}))

    def do_DELETE(self):
        headers = self.preflight()
        if headers is not None:
            self.send_reply(Reply(405, {"Allow": "POST"}, {"error": "method_not_allowed"}))

    def do_OPTIONS(self):
        # 无浏览器 CORS 功能；有效 Origin 也不会获得 CORS 授权。
        headers = self.preflight()
        if headers is not None:
            self.send_reply(Reply(405, {"Allow": "POST"}, {"error": "cors_not_implemented"}))

    def do_POST(self):
        headers = self.preflight()
        if headers is None:
            return
        if self.path != "/mcp":
            self.send_reply(Reply(404, {}, {"error": "not_found"}))
            return
        if "transfer-encoding" in headers or headers.get("expect"):
            self.send_reply(Reply(400, {}, {"error": "unsupported_framing"}))
            return
        if headers.get("content-type", "").split(";", 1)[0].strip().lower() != "application/json":
            self.send_reply(Reply(415, {}, {"error": "content_type"}))
            return
        accepts = {x.strip().lower() for x in headers.get("accept", "").split(",")}
        if not {"application/json", "text/event-stream"}.issubset(accepts):
            self.send_reply(Reply(406, {}, {"error": "accept_both_response_types"}))
            return
        length = headers.get("content-length", "")
        if not re.fullmatch(r"[0-9]{1,9}", length):
            self.send_reply(Reply(411, {}, {"error": "content_length_required"}))
            return
        size = int(length)
        if not 0 < size <= MAX_BODY:
            self.send_reply(Reply(413, {}, {"error": "request_too_large"}))
            return
        try:
            raw = self.rfile.read(size)
            if len(raw) != size:
                raise EOFError
        except (OSError, EOFError):
            self.send_reply(Reply(400, {}, {"error": "incomplete_request"}))
            return
        self.send_reply(self.app.handle(headers, raw))


class LocalService:
    def __init__(self, mode="json", port=0, clock=time.time):
        self.server = LessonHTTPServer(("127.0.0.1", port), Handler)
        self.endpoint = f"http://127.0.0.1:{self.server.server_port}/mcp"
        self.app = Application(self.endpoint, mode, clock)
        self.server.app = self.app
        self.thread = None
        self.tokens = {"reader": self.app.tokens.provision("alice", ["docs:read"]),
                       "operator": self.app.tokens.provision("alice", ["docs:read", "calc:use"]),
                       "bob": self.app.tokens.provision("bob", ["docs:read"])}

    def __enter__(self):
        self.thread = threading.Thread(target=self.server.serve_forever,
                                       kwargs={"poll_interval": 0.03}, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *_):
        if self.thread is not None:
            self.server.shutdown()
            self.thread.join(timeout=4)
        self.server.server_close()

## 5．客户端同时处理 JSON 和 SSE

客户端 `Accept` 声明同时支持 `application/json` 和 `text/event-stream`，由服务选择响应形式。SSE 必须收到匹配本次请求 ID 的最终响应；进度通知和 EOF 都不能替代它。来源：[1]。

客户端不自动升级权限、不自动刷新 token、不重发失败工具，也不跟随 3xx。这样可以把“凭证过期”“权限不够”“暂时网络失败”与业务错误分开交给宿主管理。

### 有界读取、SSE 解析与授权发现
`discover_authorization()` 实际经历 401 → 公开 metadata，但不执行登录；不会把已有访问凭证发送给 metadata 或第三方。

对本课而言，metadata URL 必须是已经配置的同源固定路径，返回的 resource 与 issuer 必须匹配预设。真实授权发现可以使用不同源，但应有相应 SSRF、发行方与 URL 验证，不能简单复制这里的窄白名单为全协议实现。来源：[3][5][6]。

In [12]:
def sse_frame(message: dict) -> bytes:
    return ("event: message\ndata: " + canonical(message) + "\n\n").encode("utf-8")


class SSEParser:
    """有限请求内 SSE。支持多 data 行、注释、LF/CRLF；EOF 不补发残帧。"""
    def __init__(self):
        self.decoder = codecs.getincrementaldecoder("utf-8")("strict")
        self.pending = ""
        self.data = []
        self.event = "message"
        self.bytes = 0
        self.count = 0

    def feed(self, chunk: bytes, final=False) -> list[dict]:
        self.bytes += len(chunk)
        require(self.bytes <= MAX_RESPONSE, "SSE 超出总字节预算")
        try:
            self.pending += self.decoder.decode(chunk, final=final)
        except UnicodeError as exc:
            raise TransportError("无效的 SSE UTF-8") from exc
        out = []
        while "\n" in self.pending:
            line, self.pending = self.pending.split("\n", 1)
            if line.endswith("\r"):
                line = line[:-1]
            if not line:
                if self.data:
                    require(self.event in ("message", ""), "不支持的 SSE event")
                    item = strict_loads("\n".join(self.data))
                    require(isinstance(item, dict), "SSE data 必须是 JSON 对象")
                    out.append(item)
                    self.count += 1
                    require(self.count <= MAX_EVENTS, "SSE 事件预算耗尽")
                self.data, self.event = [], "message"
                continue
            if line.startswith(":"):
                continue
            field, _, value = line.partition(":")
            if value.startswith(" "):
                value = value[1:]
            if field == "data":
                self.data.append(value)
            elif field == "event":
                self.event = value
            # SSE 其他字段忽略；本课不支持 Last-Event-ID 恢复，也不重连。
        return out


def collect_sse(messages: list[dict], expected_id, progress_token: str | None = None) -> tuple[dict, int]:
    final = None
    progress_count = 0
    for msg in messages:
        require(final is None, "最终 RPC 响应后还有数据事件")
        if msg.get("method") == "notifications/progress" and "id" not in msg:
            p = msg.get("params")
            require(msg.get("jsonrpc") == "2.0" and isinstance(p, dict)
                    and progress_token is not None and p.get("progressToken") == progress_token,
                    "进度通知没有关联到本次请求")
            progress_count += 1
        else:
            require("method" not in msg and "id" in msg, "SSE 中出现独立服务端请求或非法消息")
            # validate_reply 会校验 id/resultType，业务 isError 在更上层处理。
            final = msg
    require(final is not None, "EOF 之前没有收到最终 RPC 响应")
    validate_reply(final, expected_id)
    return final, progress_count


def validate_resource_metadata(document: dict, resource: str, trusted_issuers: set[str]) -> dict:
    require(isinstance(document, dict) and document.get("resource") == resource, "resource 不匹配")
    issuers = document.get("authorization_servers")
    require(isinstance(issuers, list) and len(issuers) > 0 and len(set(issuers)) == len(issuers), "授权服务列表错误")
    require(all(isinstance(x, str) and x in trusted_issuers for x in issuers), "发现了尚未信任的 issuer")
    require(document.get("bearer_methods_supported") == ["header"], "本课只允许 Authorization header")
    return copy.deepcopy(document)


def metadata_url_from_challenge(challenge: str, endpoint: str) -> str:
    require(isinstance(challenge, str) and challenge.startswith("Bearer "), "不是本课 Bearer challenge")
    found = re.findall(r'(?:^|[ ,])resource_metadata="([^"\r\n]+)"', challenge)
    require(len(found) == 1, "缺少或重复 resource_metadata")
    expected = endpoint[:-4] + METADATA_PATH
    require(found[0] == expected, "本课仅获取已配置同源 metadata 路径；不跟随任意发现 URL")
    return found[0]


class HttpClient:
    def __init__(self, endpoint: str, token: str | None = None, allowed_tools=None, budget=20):
        p = endpoint_parts(endpoint)
        self.endpoint, self.port, self.origin = endpoint, p.port, endpoint[:-4]
        if token is not None:
            require(isinstance(token, str) and re.fullmatch(r"[A-Za-z0-9_.~-]{1,256}", token) is not None,
                    "开发凭证必须是有界、不含控制字符的字符串")
        self.token = token
        self.allowed_tools = set(TOOL_SCHEMAS) if allowed_tools is None else set(allowed_tools)
        require(self.allowed_tools.issubset(TOOL_SCHEMAS), "宿主白名单含未知工具")
        require(type(budget) is int and budget >= 1, "请求预算错误")
        self.budget, self.sent, self.next_id, self.wire = budget, 0, 1, []

    def exchange(self, method: str, path: str, headers: dict, data: bytes | None = None,
                 expected_id=None, progress_token=None) -> dict:
        require(self.sent < self.budget, "客户端请求预算耗尽")
        require(path.startswith("/") and "\r" not in path and "\n" not in path, "path 错误")
        self.sent += 1
        connection = http.client.HTTPConnection("127.0.0.1", self.port, timeout=3.0)
        start = time.monotonic()
        try:
            connection.request(method, path, body=data, headers=headers)
            response = connection.getresponse()
            rh = {k.lower(): v for k, v in response.getheaders()}
            status = response.status
            if 300 <= status < 400:
                # http.client 本身不跟随；显式拒绝，不把凭证带到 Location。
                raise TransportError("重定向被拒绝；不转发 Authorization")
            ctype = rh.get("content-type", "").split(";", 1)[0].lower()
            chunks, size, parser, events = [], 0, SSEParser(), []
            while True:
                require(time.monotonic() - start <= 8.0, "响应读取总期限超出")
                chunk = response.read1(512)
                if not chunk:
                    break
                size += len(chunk)
                require(size <= MAX_RESPONSE, "响应过大")
                if ctype == "text/event-stream" and 200 <= status < 300:
                    events.extend(parser.feed(chunk))
                else:
                    chunks.append(chunk)
            if ctype == "text/event-stream" and status == 200:
                events.extend(parser.feed(b"", final=True))
                reply, count = collect_sse(events, expected_id, progress_token)
                body = reply
            else:
                require(ctype == "application/json", "不支持的响应类型")
                body = strict_loads(b"".join(chunks).decode("utf-8"))
                count = 0
            # 审计只有固定路径、状态、RPC id、内容摘要；没有 Authorization 或 token。
            self.wire.append({"sequence": self.sent, "method": method, "path": path.split("?", 1)[0],
                              "status": status, "content_type": ctype, "request_id": expected_id,
                              "progress_notifications": count, "body_sha256": digest(body)})
            return {"status": status, "headers": rh, "body": body, "progress_count": count}
        except (OSError, http.client.HTTPException, UnicodeError) as exc:
            raise TransportError("HTTP 传输失败；未自动重试") from exc
        finally:
            connection.close()

    def raw_rpc(self, body: dict, header_changes: dict | None = None) -> dict:
        headers = request_headers(body)
        if self.token is not None:
            headers["Authorization"] = "Bearer " + self.token
        for key, value in (header_changes or {}).items():
            if value is None:
                headers.pop(key, None)
            else:
                headers[key] = value
        return self.exchange("POST", "/mcp", headers, canonical(body).encode("utf-8"),
                             body["id"], body["params"]["_meta"].get("progressToken"))

    def request(self, method: str, params=None, progress=False) -> dict:
        rid = self.next_id
        self.next_id += 1
        body = make_request(method, params, rid, progress_token=f"progress-{rid}" if progress else None)
        response = self.raw_rpc(body)
        if response["status"] != 200:
            content = response["body"]
            if isinstance(content.get("error"), dict) and "jsonrpc" in content:
                e = content["error"]
                raise RPCFailure(e["code"], e["message"], response["status"])
            raise HTTPFailure(response["status"], content.get("error", "unknown"), response["headers"])
        return validate_reply(response["body"], rid)

    def call_tool(self, name: str, arguments: dict) -> dict:
        require(name in self.allowed_tools, "宿主没有允许该工具；尚未发送 HTTP")
        result = self.request("tools/call", {"name": name, "arguments": arguments}, progress=True)
        return validate_tool_result(result)

    def discover_authorization(self) -> dict:
        # 从实际 401 获取挑战；此探测不携带已有凭证。
        probe = make_request("server/discover", request_id="auth-probe")
        r = self.exchange("POST", "/mcp", request_headers(probe), canonical(probe).encode(), "auth-probe")
        require(r["status"] == 401, "预期未经认证的 challenge")
        url = metadata_url_from_challenge(r["headers"].get("www-authenticate"), self.endpoint)
        path = urlsplit(url).path
        metadata = self.exchange("GET", path, {"Accept": "application/json"})
        require(metadata["status"] == 200, "metadata 请求失败")
        return validate_resource_metadata(metadata["body"], self.endpoint, {ISSUER})

In [13]:
with LocalService("json") as service:
    public_client = HttpClient(service.endpoint)
    metadata = public_client.discover_authorization()
    print("从真实401发现 metadata：", metadata["resource"] == service.endpoint)
    print("发行方标识：", metadata["authorization_servers"])
    print("已完成OAuth登录：False")
    reader = HttpClient(service.endpoint, service.tokens["reader"])
    operator = HttpClient(service.endpoint, service.tokens["operator"])
    print("reader工具：", [t["name"] for t in reader.request("tools/list")["tools"]])
    print("operator工具：", [t["name"] for t in operator.request("tools/list")["tools"]])
    print("公开metadata请求没有凭证：",
          any(e["outcome"] == "metadata_without_authorization" for e in service.app.audit))

从真实401发现 metadata： True
发行方标识： ['https://auth.lesson52.example']
已完成OAuth登录：False
reader工具： ['read_doc']
operator工具： ['calculate', 'read_doc']
公开metadata请求没有凭证： True


### 401、403、RPC error、工具错误，分别处理

| 结果 | 本课含义 |
|---|---|
| 401 | 缺少、未知、过期、撤销、issuer 或 audience 不匹配的凭证 |
| 403 | 凭证有效，但缺少 scope 或对象权限；也可能是非法 Origin |
| 400 + `-32020` | 请求头与正文不一致 |
| 404 + `-32601` | 不支持的方法 |
| 200 + JSON-RPC error | 协议/调用外壳错误，例如未知工具 |
| 200 + `result.isError=true` | 请求完整返回，但工具业务失败，例如除零 |

401/403 与 Bearer challenge 的规定见 [3]；工具业务失败与协议错误的区别见 [4]。

不要一遇到 403 就无限请求更大权限。生产中的 step-up 必须走有效授权流程及重试预算；本课没有自动 step-up。

In [14]:
with LocalService() as service:
    reader = HttpClient(service.endpoint, service.tokens["reader"])
    before = service.app.executions
    try:
        reader.call_tool("calculate", {"op": "subtract", "left": "30", "right": "14"})
    except HTTPFailure as error:
        print("reader计算：", error.status, error.code)
    print("权限拒绝后工具执行增量：", service.app.executions - before)
    operator = HttpClient(service.endpoint, service.tokens["operator"])
    failed = operator.call_tool("calculate", {"op": "divide", "left": "1", "right": "0"})
    print("operator除零：", failed)
    print("除零是实际工具业务错误，不是身份错误。")

reader计算： 403 insufficient_scope
权限拒绝后工具执行增量： 0
operator除零： {'data': None, 'error': 'division_by_zero', 'ok': False}
除零是实际工具业务错误，不是身份错误。


### 伪装客户端名称，没有获得更多权限

接下来的请求把 `_meta.clientInfo.name` 改为 `administrator`，但仍持有 reader 凭证。身份来自经过校验的服务端凭证记录，不来自模型可控文本。

此外，目录过滤只是便利性：即使调用方事先知道计算器名称，直接调用仍需检查权限。

In [15]:
with LocalService() as service:
    reader = HttpClient(service.endpoint, service.tokens["reader"])
    message = make_request("tools/call",
        {"name": "calculate", "arguments": {"op": "add", "left": "1", "right": "2"}},
        client_name="administrator")
    reply = reader.raw_rpc(message)
    print({"http_status": reply["status"], "executions": service.app.executions})
    bob = HttpClient(service.endpoint, service.tokens["bob"])
    try:
        bob.call_tool("read_doc", {"doc_id": "alice_private"})
    except HTTPFailure as error:
        print("bob读取alice资料：", error.status, error.code)

{'http_status': 403, 'executions': 0}
bob读取alice资料： 403 object_forbidden


### 服务名称不是服务身份

下面两份独立服务故意使用完全一样的 `serverInfo`。名字只能作为标签，不能触发“这是管理员服务，所以信任它”的行为。

实际远端服务器的网络身份由正确的 TLS/证书验证等机制建立；资源授权仍要检查 token 是否发给该 resource。Bearer 凭证也不能被直接转发给另一个下游服务。来源：[2][3][5]。

In [16]:
with LocalService() as first, LocalService() as second:
    a = HttpClient(first.endpoint, first.tokens["reader"])
    b = HttpClient(second.endpoint, second.tokens["reader"])
    info_a = a.request("server/discover")["_meta"][META + "serverInfo"]
    info_b = b.request("server/discover")["_meta"][META + "serverInfo"]
    print("名称与版本相同：", info_a == info_b)
    print("服务地址不同：", first.endpoint != second.endpoint)
    wrong_target = HttpClient(second.endpoint, first.tokens["reader"])
    try:
        wrong_target.request("tools/list")
    except HTTPFailure as error:
        print("跨服务复用第一份开发凭证：", error.status)

名称与版本相同： True
服务地址不同： True
跨服务复用第一份开发凭证： 401


## 6．协议头错配必须在执行前拦住

先保持合法 operator 凭证，再把 `Mcp-Name` 头伪装为只读文档工具，而正文仍请求计算器。与直接 scope 检查不同，这次测的是网关与后端可能发生的“两个来源不一致”。

In [17]:
with LocalService() as service:
    client = HttpClient(service.endpoint, service.tokens["operator"])
    body = make_request("tools/call",
                        {"name": "calculate", "arguments": {"op": "subtract", "left": "30", "right": "14"}})
    reply = client.raw_rpc(body, {"Mcp-Name": "read_doc"})
    print("HTTP：", reply["status"])
    print("RPC：", reply["body"]["error"])
    print("实际工具执行：", service.app.executions)

HTTP： 400
RPC： {'code': -32020, 'message': 'HeaderMismatch'}
实际工具执行： 0


## 7．JSON 与 SSE 的计算含义保持一致

本课服务端对发现和目录使用 JSON；SSE 模式下，对工具结果先发送关联进度，再发送最终 RPC 响应。没有模型 token，没有回复生成，也没有服务端独立发起的 RPC。

SSE 采用 UTF-8，网络分段不是事件边界。流被切断而未收到最终响应时，应报告不完整；不能悄悄重发可能已执行的工具。来源：[1][7]。

In [18]:
transport_observations = []
for mode in ("json", "sse"):
    with LocalService(mode) as service:
        client = HttpClient(service.endpoint, service.tokens["operator"])
        outcome = client.call_tool("calculate", {"op": "subtract", "left": "30", "right": "14"})
        transport_observations.append({"mode": mode, "value": outcome["data"]["value"],
            "media_type": client.wire[-1]["content_type"],
            "progress_notifications": client.wire[-1]["progress_notifications"]})
print(json.dumps(transport_observations, ensure_ascii=False, indent=2))

[
  {
    "mode": "json",
    "value": "16",
    "media_type": "application/json",
    "progress_notifications": 0
  },
  {
    "mode": "sse",
    "value": "16",
    "media_type": "text/event-stream",
    "progress_notifications": 1
  }
]


In [19]:
final_message = rpc_result(7, {"text": "你好🌍"})
encoded = sse_frame(final_message)
parser = SSEParser()
received = []
for byte in encoded:
    received.extend(parser.feed(bytes([byte])))
received.extend(parser.feed(b"", final=True))
print("逐字节喂入仍还原：", received == [final_message])
print("最终id：", collect_sse(received, 7)[0]["id"])
partial = SSEParser().feed(encoded[:-1], final=True)
print("没有终止空行的事件：", partial)
try:
    collect_sse(partial, 7)
except ContractError as error:
    print("EOF不能假装完成：", str(error))

逐字节喂入仍还原： True
最终id： 7
没有终止空行的事件： []
EOF不能假装完成： EOF 之前没有收到最终 RPC 响应


### 与第 49、50 课的重连机制区分

那些课程使用自定义事件日志和重放接口。当前 MCP 修订版不提供 `Last-Event-ID` 恢复；GET 旧监听端点也已经移除。本课按版本约定忽略旧 session/重放头，不把它们当成身份或重新执行依据。客户端不会自动重连或重试。来源：[1]。

当前工具很短且只读。连接失败时，它们可能已经完成；关闭 socket 也不能撤销完成的计算，更不能回滚未来的写操作。

## 8．拒绝重定向，避免把凭证带去未知目的地

正式客户端要验证发现 URL、issuer、redirect_uri 与目标资源，防止 token 泄露或 SSRF。本课更保守：只访问固定回环 endpoint 和同源 metadata，所有 3xx 都中止，不跟随 `Location`，不自动升级权限。

下面真正启动两台本地 HTTP 服务：第一台返回重定向，第二台计数是否收到请求。没有向外部网站发送任何数据。来源：[5][6]。

### 重定向实验定义
注意：本函数没有攻击外部服务，也不收集其他人的凭证。所有数据均由课堂自行创建，工具只统计本机请求次数。

In [20]:
def redirect_experiment() -> dict:
    counters = {"source":0, "destination":0}
    class Sink(BaseHTTPRequestHandler):
        def log_message(self,*_): pass
        def do_POST(self):
            counters["destination"] += 1
            self.send_response(204); self.end_headers()
    sink = LessonHTTPServer(("127.0.0.1",0),Sink)
    class Redirect(BaseHTTPRequestHandler):
        def log_message(self,*_): pass
        def do_POST(self):
            counters["source"] += 1
            self.send_response(302)
            self.send_header("Location",f"http://127.0.0.1:{sink.server_port}/mcp")
            self.send_header("Content-Length","0")
            self.end_headers()
    source = LessonHTTPServer(("127.0.0.1",0),Redirect)
    threads = []
    for server in (sink,source):
        th = threading.Thread(target=server.serve_forever,kwargs={"poll_interval":0.02},daemon=True)
        th.start(); threads.append(th)
    client = HttpClient(f"http://127.0.0.1:{source.server_port}/mcp","development-not-real-secret")
    blocked = False
    try:
        client.request("tools/list")
    except TransportError:
        blocked = True
    finally:
        for server in (source,sink): server.shutdown(); server.server_close()
        for th in threads: th.join(timeout=3)
    return {"blocked":blocked,"source_requests":counters["source"],
            "destination_requests":counters["destination"],"client_requests":client.sent}

In [21]:
redirect_report = redirect_experiment()
print(json.dumps(redirect_report, ensure_ascii=False, indent=2))

{
  "blocked": true,
  "source_requests": 1,
  "destination_requests": 0,
  "client_requests": 1
}


## 9．每个请求重新验证：到期或撤销不能因为“已经连接过”而绕过

本例使用受控时钟，不等待真实的几分钟。第一次成功读取后，把时间推进到有效期边界；下一次请求必须失败。正式访问 token 的过期、目标资源与 scope 检查要求见 [3]。

开发凭证在进程重启后也会失效。本课没有 refresh token：需要重新启动服务并在可信终端取得本次凭证，不要在模型提示里粘贴它。

In [22]:
clock = [100.0]
with LocalService(clock=lambda: clock[0]) as service:
    token = service.app.tokens.provision("alice", ["docs:read"], ttl=5)
    client = HttpClient(service.endpoint, token)
    print("到期前：", client.call_tool("read_doc", {"doc_id": "backup"})["ok"])
    clock[0] = 105.0
    try:
        client.call_tool("read_doc", {"doc_id": "backup"})
    except HTTPFailure as error:
        print("到期边界：", error.status, error.code)
    print("成功进入工具的次数：", service.app.executions)

到期前： True
到期边界： 401 invalid_token
成功进入工具的次数： 1


## 10．完整实验：同一业务流程，两种传输；权限错误均不执行

读取两份虚构资料，从原文提取数字，再调用计算器。两条传输路径都使用五次 HTTP 请求：发现一次、列目录一次、三次业务工具调用。

随后运行身份与权限矩阵：无凭证、过期、已撤销、错误发行方、错误目标、缺少 scope、对象归属、Origin 与镜像头冲突。所有预期拒绝都检查工具执行计数，而不只看错误文字。

### 完整实验定义
报告不保存随机 token；审计只记录主体、固定方法名、工具名和结果类别，不记录 Authorization 或任意用户正文。开发服务终端打印临时凭证是显式的本机教学入口，不能照搬成生产日志。

In [23]:
def run_experiment() -> dict:
    transports = {}
    for mode in ("json", "sse"):
        with LocalService(mode) as service:
            client = HttpClient(service.endpoint, service.tokens["operator"])
            discovery = client.request("server/discover")
            catalog = client.request("tools/list")
            backup = client.call_tool("read_doc", {"doc_id": "backup"})
            trash = client.call_tool("read_doc", {"doc_id": "trash"})
            a = re.search(r"(\d+) 天", trash["data"]["text"]).group(1)
            b = re.search(r"(\d+) 天", backup["data"]["text"]).group(1)
            calculation = client.call_tool("calculate", {"op": "subtract", "left": a, "right": b})
            transports[mode] = {"protocol": discovery["supportedVersions"],
                "tools": [t["name"] for t in catalog["tools"]],
                "draft": f"回收站保留{a}天，自动备份保留{b}天，前者多{calculation['data']['value']}天。",
                "tool_executions": service.app.executions, "http_requests": client.sent,
                "progress_events": sum(w["progress_notifications"] for w in client.wire),
                "wire": client.wire, "task_success": None, "semantic_support_checked": False}
    with LocalService() as service:
        app = service.app
        roles = {name: HttpClient(service.endpoint, token, budget=60) for name, token in service.tokens.items()}
        metadata = HttpClient(service.endpoint).discover_authorization()
        matrix = []

        def add(label, token, tool, arguments, header_changes=None, client_name="lesson52-client"):
            client = HttpClient(service.endpoint, token)
            before = app.executions
            body = make_request("tools/call", {"name": tool, "arguments": arguments}, client_name=client_name)
            r = client.raw_rpc(body, header_changes)
            matrix.append({"scenario": label, "status": r["status"],
                           "executed": app.executions - before,
                           "rpc_code": r["body"].get("error", {}).get("code") if isinstance(r["body"].get("error"), dict) else None,
                           "is_tool_error": r["body"].get("result", {}).get("isError")})

        read = {"doc_id": "backup"}
        calc = {"op": "subtract", "left": "30", "right": "14"}
        add("无凭证", None, "read_doc", read)
        add("未知凭证", "not-issued", "read_doc", read)
        add("过期凭证", app.tokens.provision("alice", ["docs:read"], ttl=-1), "read_doc", read)
        add("目标资源不匹配", app.tokens.provision("alice", ["docs:read"], audience="https://other.example/mcp"), "read_doc", read)
        add("发行方不匹配", app.tokens.provision("alice", ["docs:read"], issuer="https://other.example"), "read_doc", read)
        revoked = app.tokens.provision("alice", ["docs:read"])
        app.tokens.revoke(revoked)
        add("已撤销凭证", revoked, "read_doc", read)
        add("reader读取公共资料", service.tokens["reader"], "read_doc", read)
        add("reader越权计算", service.tokens["reader"], "calculate", calc)
        add("伪装clientInfo为管理员", service.tokens["reader"], "calculate", calc, client_name="administrator")
        add("operator计算", service.tokens["operator"], "calculate", calc)
        add("bob读取alice对象", service.tokens["bob"], "read_doc", {"doc_id": "alice_private"})
        add("alice读取自己的对象", service.tokens["reader"], "read_doc", {"doc_id": "alice_private"})
        add("错误Origin", service.tokens["operator"], "calculate", calc, {"Origin": "https://untrusted.example"})
        add("头与正文工具名冲突", service.tokens["operator"], "calculate", calc, {"Mcp-Name": "read_doc"})
        add("合法除零业务失败", service.tokens["operator"], "calculate", {"op": "divide", "left": "1", "right": "0"})
        host = HttpClient(service.endpoint, service.tokens["operator"], allowed_tools={"read_doc"})
        try:
            host.call_tool("calculate", calc)
        except ContractError:
            host_blocked = host.sent == 0
        else:
            host_blocked = False
        catalogs = {name: [t["name"] for t in c.request("tools/list")["tools"]] for name, c in roles.items()}
        # 不把随机 endpoint/token 写入核心结果；保留公开身份字段的可解释摘要。
        identity = {"challenge_status": 401, "metadata_resource_matches_endpoint": metadata["resource"] == service.endpoint,
                    "authorization_servers": metadata["authorization_servers"], "real_oauth_login": False,
                    "catalogs": catalogs, "cases": matrix, "host_denial_sent_requests": host.sent,
                    "host_denial_verified": host_blocked, "audit": copy.deepcopy(app.audit)}
        secret_free = canonical({"transports": transports, "identity": identity})
        require(all(t not in secret_free for t in service.tokens.values()), "报告意外包含开发凭证")
    return {"mode": "real_loopback_http_with_development_credentials", "protocol": PROTOCOL,
            "transports": transports, "authorization": identity,
            "limits": ["没有 OAuth 登录/PKCE/refresh/JWT验签", "没有TLS或公网部署", "不是完整MCP兼容认证",
                       "没有模型、写工具、MRTR、订阅或重放", "不把serverInfo当认证身份", "开发凭证只在当前服务进程有效"]}

In [24]:
experiment = run_experiment()
for mode, result in experiment["transports"].items():
    print(mode, result["draft"])
    print("HTTP请求：", result["http_requests"], "工具执行：", result["tool_executions"],
          "进度事件：", result["progress_events"])
print("本课没有调用模型；草稿仍未完成语义支持审核。")

json 回收站保留30天，自动备份保留14天，前者多16天。
HTTP请求： 5 工具执行： 3 进度事件： 0
sse 回收站保留30天，自动备份保留14天，前者多16天。
HTTP请求： 5 工具执行： 3 进度事件： 3
本课没有调用模型；草稿仍未完成语义支持审核。


In [25]:
for row in experiment["authorization"]["cases"]:
    print(f"{row['scenario']:<18} HTTP={row['status']} 执行={row['executed']} "
          f"RPC={row['rpc_code']} isError={row['is_tool_error']}")
print("宿主拒绝发送次数：", experiment["authorization"]["host_denial_sent_requests"])

无凭证                HTTP=401 执行=0 RPC=None isError=None
未知凭证               HTTP=401 执行=0 RPC=None isError=None
过期凭证               HTTP=401 执行=0 RPC=None isError=None
目标资源不匹配            HTTP=401 执行=0 RPC=None isError=None
发行方不匹配             HTTP=401 执行=0 RPC=None isError=None
已撤销凭证              HTTP=401 执行=0 RPC=None isError=None
reader读取公共资料       HTTP=200 执行=1 RPC=None isError=False
reader越权计算         HTTP=403 执行=0 RPC=None isError=None
伪装clientInfo为管理员   HTTP=403 执行=0 RPC=None isError=None
operator计算         HTTP=200 执行=1 RPC=None isError=False
bob读取alice对象       HTTP=403 执行=0 RPC=None isError=None
alice读取自己的对象       HTTP=200 执行=1 RPC=None isError=False
错误Origin           HTTP=403 执行=0 RPC=None isError=None
头与正文工具名冲突          HTTP=400 执行=0 RPC=-32020 isError=None
合法除零业务失败           HTTP=200 执行=1 RPC=None isError=True
宿主拒绝发送次数： 0


## 11．完整 OAuth 还缺哪些环节？

真正的部署大致有以下过程：

```text
请求受保护资源 → 401 与 resource metadata
  → 验证资源/授权服务器身份
  → 选择客户端登记方式并发起授权
  → 用户交互、授权码与 PKCE 等协议检查
  → 获得面向该 resource 的访问 token
  → MCP服务验证 token 和 scope
  → 过期或权限不足时，按正规流程刷新或重新授权
```

**本课只运行前面的挑战/metadata和后面的资源访问校验，跳过中间真正的签发流程。** `https://auth.lesson52.example` 是保留的示例标识，不存在可供本课登录的端点。本课 token 表不是 OAuth 服务，也不是 JWT/JWKS 验证器。

模型不接触 Authorization 头；即使之后增加下游 API，也不能把收到的 MCP token 原样转发给下游当作其凭证。需要正确的目标资源授权与委托设计。来源：[3][5][6]。

## 12．独立练习

先补全下面的函数，再打开各自检查开关。默认运行全部单元只会跳过练习，不会伪装成已经完成。

练习 A 要核对版本、方法、工具名，包括编码后的名称；练习 B 接收的是**服务端可信记录**，不是客户端声称的 JSON；练习 C 只负责完整消息收集，工具业务是否成功仍需由上层检查 `isError`。

In [26]:
def exercise_mirrors(body, headers):
    raise NotImplementedError("练习A：检查三个镜像头，支持 Mcp-Name Base64 编码")


def exercise_authorize(record, resource, issuer, now, scope):
    raise NotImplementedError("练习B：校验可信记录的 issuer/audience/time，再判断 scope")


def exercise_collect(messages, expected_id, progress_token=None):
    raise NotImplementedError("练习C：进度不算完成，最终响应必须存在且 id 正确")


class Checks:
    def __init__(self):
        self.names = []

    def yes(self, name, condition):
        if not condition:
            raise AssertionError(name)
        self.names.append(name)

    def eq(self, name, actual, expected):
        self.yes(name, actual == expected)

    def raises(self, name, kind, fn):
        try:
            fn()
        except kind:
            self.names.append(name)
        else:
            raise AssertionError(name + "：应拒绝而未拒绝")


def exercise_checks(which: str, fn) -> int:
    t = Checks()
    if which == "A":
        body = make_request("tools/call", {"name": "read_doc", "arguments": {"doc_id": "backup"}})
        headers = {k.lower(): v for k, v in request_headers(body).items()}
        t.eq("A正确镜像", fn(body, headers), None)
        for h in ("mcp-method", "mcp-name", "mcp-protocol-version"):
            wrong = dict(headers, **{h: "wrong"})
            t.raises("A拒绝" + h, ContractError, lambda w=wrong: fn(body, w))
        body["params"]["name"] = "资料读取"
        headers["mcp-name"] = encode_header("资料读取")
        t.eq("A处理编码名称", fn(body, headers), None)
    elif which == "B":
        r = {"subject": "alice", "scopes": ["docs:read"], "issuer": ISSUER,
             "audience": "resource-A", "not_before": 0, "expires_at": 100, "revoked": False}
        t.eq("B合法授权", fn(r, "resource-A", ISSUER, 99, "docs:read"), "alice")
        for key, value in (("audience", "resource-B"), ("issuer", "bad"), ("revoked", True), ("expires_at", 99)):
            t.raises("B拒绝" + key, GrantRejected, lambda k=key, v=value: fn({**r, k:v}, "resource-A", ISSUER, 99, "docs:read"))
        t.raises("B拒绝超scope", PermissionError, lambda: fn(r, "resource-A", ISSUER, 1, "calc:use"))
    elif which == "C":
        final = rpc_result(7, {})
        p = {"jsonrpc": "2.0", "method": "notifications/progress", "params": {"progressToken": "p", "progress": 1}}
        t.eq("C进度加结果", fn([p, final], 7, "p"), (final, 1))
        t.raises("C只有进度不完整", ContractError, lambda: fn([p], 7, "p"))
        t.raises("C错误id", ContractError, lambda: fn([final], 8, "p"))
        t.raises("C重复最终结果", ContractError, lambda: fn([final, final], 7))
        t.raises("C进度串线", ContractError, lambda: fn([p, final], 7, "q"))
    else:
        raise ValueError("练习只能为A/B/C")
    return len(t.names)

### 练习 A：镜像头一致性

实现 `exercise_mirrors(body, headers)`，合法时返回 `None`，不匹配时抛出 `ContractError`。传入的头名称已小写。不得修改输入。

In [27]:
RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print("A通过：", exercise_checks("A", exercise_mirrors))
else:
    print("练习A尚待独立完成。")

练习A尚待独立完成。


### 练习 B：凭证有效性与 scope

实现 `exercise_authorize(...)`：先验证 issuer、audience、有效时间、撤销状态，再检查 scope。成功返回 subject；凭证失效抛 `GrantRejected`，权限不够抛 `PermissionError`。

注意有效区间为 `not_before <= now < expires_at`。不能把 scope 合法作为忽略 audience 的理由。

In [28]:
RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print("B通过：", exercise_checks("B", exercise_authorize))
else:
    print("练习B尚待独立完成。")

练习B尚待独立完成。


### 练习 C：收集请求内 SSE

实现 `exercise_collect(messages, expected_id, progress_token=None)`，返回 `(最终RPC消息, 进度数量)`。只有进度、错误 ID、重复最终结果或串线的进度都不能通过。

请解释：`resultType="complete"` 能否推出 `isError=False`？

In [29]:
RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print("C通过：", exercise_checks("C", exercise_collect))
else:
    print("练习C尚待独立完成。")

练习C尚待独立完成。


## 13．参考实现

下面三个参考函数复用本课前面已经展开的核心实现，以便逐行对照，而不覆盖你的练习函数。先完成自己的版本，再检查输出。

In [30]:
def reference_mirrors(body, headers):
    return validate_mirrors(body, headers)

In [31]:
def reference_authorize(record, resource, issuer, now, scope):
    p = validate_grant(record, resource, issuer, now)
    if scope not in p.scopes:
        raise PermissionError("insufficient_scope")
    return p.subject

In [32]:
def reference_collect(messages, expected_id, progress_token=None):
    return collect_sse(messages, expected_id, progress_token)

In [33]:
for name, fn in [("A", reference_mirrors), ("B", reference_authorize), ("C", reference_collect)]:
    print(name, "参考实现检查：", exercise_checks(name, fn))

A 参考实现检查： 5
B 参考实现检查： 6
C 参考实现检查： 5


## 14．完整自动检查与故障注入

检查既包含纯函数，也会实际启动本地 HTTP 服务，验证 401/403、对象权限、重复 Authorization、镜像错误、JSON/SSE、超时预算与重定向拒绝。

这不是渗透测试或协议认证；只证明本课列出的情况。官方 SDK 互操作、TLS、真实登录、代理和生产并发未测试。

### 完整检查定义
可折叠阅读。测试包含实际网络返回状态与执行计数，不只验证异常字符串。

In [34]:
def run_checks() -> dict:
    t = Checks()
    t.raises("JSON重复键", ContractError, lambda: strict_loads('{"x":1,"x":2}'))
    t.raises("JSON NaN", ContractError, lambda: strict_loads('{"x":NaN}'))
    t.raises("JSON极大指数", ContractError, lambda: strict_loads('1e999'))
    t.eq("Unicode往返", strict_loads(canonical({"x": "中文🌍"})), {"x": "中文🌍"})
    for endpoint in ("http://evil.example/mcp", "http://u:p@127.0.0.1:8000/mcp", "http://127.0.0.1:8000/mcp?token=x"):
        t.raises("拒绝非固定来源" + endpoint, ContractError, lambda x=endpoint: endpoint_parts(x))
    for value in ("read_doc", "资料读取", " a ", "=?base64?literal?=", "a\nb"):
        t.eq("镜像编码往返" + repr(value), decode_header(encode_header(value)), value)
    t.raises("损坏Base64", ContractError, lambda: decode_header("=?base64?!!!?="))
    t.raises("明文换行头", ContractError, lambda: decode_header("a\nb"))
    t.raises("布尔RPC id", ContractError, lambda: make_request("tools/list", request_id=True))
    t.raises("空数组不是params对象", ContractError, lambda: make_request("tools/list", []))
    p = {"name": "read_doc", "arguments": {"doc_id": "backup"}}
    before = copy.deepcopy(p)
    body = make_request("tools/call", p)
    t.eq("构造请求不改输入", p, before)
    t.eq("resultType complete", validate_reply(rpc_result(1, {}), 1)["resultType"], "complete")
    t.raises("响应id错误", ContractError, lambda: validate_reply(rpc_result(2, {}), 1))
    t.raises("RPC失败不是成功", RPCFailure, lambda: validate_reply(rpc_error(1, -32602, "bad"), 1))
    failed = {"resultType": "complete", **tool_result(error="division_by_zero")}
    t.eq("complete可伴随业务失败", validate_tool_result(failed)["ok"], False)
    t.raises("结果布尔类型", ContractError, lambda: validate_tool_result({**failed, "isError": 1}))
    t.raises("禁止表达式", ContractError, lambda: validate_arguments("calculate", {"op":"add", "left":"1+2", "right":"3"}))
    t.raises("禁止身份参数", ContractError, lambda: validate_arguments("read_doc", {"doc_id":"backup", "subject":"admin"}))
    for label, fn in (("A", reference_mirrors), ("B", reference_authorize), ("C", reference_collect)):
        t.yes("练习" + label + "参考检查", exercise_checks(label, fn) >= 5)
    record = {"subject":"alice", "scopes":["docs:read"], "issuer":ISSUER, "audience":"r", "expires_at":100, "not_before":1, "revoked":False}
    t.raises("尚未生效", GrantRejected, lambda: validate_grant(record,"r",ISSUER,0))
    t.raises("过期边界", GrantRejected, lambda: validate_grant(record,"r",ISSUER,100))
    t.eq("有效边界", validate_grant(record,"r",ISSUER,1).subject, "alice")
    t.raises("issuer不能自行补斜杠", GrantRejected, lambda: validate_grant({**record,"issuer":ISSUER+"/"},"r",ISSUER,2))
    final = rpc_result(4, {"content": [{"type":"text", "text":"中文🌍"}]})
    wire = b": heartbeat\r\n\r\n" + sse_frame(final)
    parser = SSEParser()
    events = []
    for b in wire:
        events.extend(parser.feed(bytes([b])))
    events.extend(parser.feed(b"", final=True))
    t.eq("逐字节SSE Unicode", events, [final])
    parser = SSEParser()
    t.eq("残缺SSE不能派发", parser.feed(sse_frame(final)[:-1], final=True), [])
    t.raises("EOF不算完成", ContractError, lambda: collect_sse([], 4))
    t.raises("SSE无效UTF8", TransportError, lambda: SSEParser().feed(b"\xff"))
    t.raises("SSE响应过大", ContractError, lambda: SSEParser().feed(b"x"*(MAX_RESPONSE+1)))
    goodmeta = {"resource":"http://127.0.0.1:9000/mcp", "authorization_servers":[ISSUER], "bearer_methods_supported":["header"]}
    t.eq("metadata身份一致", validate_resource_metadata(goodmeta, goodmeta["resource"], {ISSUER}), goodmeta)
    t.raises("metadata资源错配", ContractError, lambda: validate_resource_metadata(goodmeta,"different",{ISSUER}))
    t.raises("不信任发现issuer", ContractError, lambda: validate_resource_metadata({**goodmeta,"authorization_servers":["https://other.example"]},goodmeta["resource"],{ISSUER}))
    t.raises("发现URL不任意获取", ContractError, lambda: metadata_url_from_challenge('Bearer resource_metadata="http://169.254.169.254/"',goodmeta["resource"]))

    with LocalService("json") as s:
        operator = HttpClient(s.endpoint, s.tokens["operator"], budget=100)
        reader = HttpClient(s.endpoint, s.tokens["reader"], budget=100)
        bob = HttpClient(s.endpoint, s.tokens["bob"], budget=100)
        r = HttpClient(s.endpoint).raw_rpc(make_request("tools/list"))
        t.eq("无token真实HTTP401", r["status"], 401)
        t.yes("401提供metadata挑战", "resource_metadata=" in r["headers"]["www-authenticate"])
        metadata = operator.discover_authorization()
        t.eq("真实发现resource", metadata["resource"], s.endpoint)
        t.yes("公开发现不带Authorization", any(x["outcome"] == "metadata_without_authorization" for x in s.app.audit))
        t.yes("服务只保存token摘要", all(token not in canonical(s.app.tokens._records) for token in s.tokens.values()))
        t.eq("发现不获得token", metadata["lesson52_mode"], "preprovisioned_opaque_development_credentials_no_oauth_login")
        t.eq("operator目录", [x["name"] for x in operator.request("tools/list")["tools"]], ["calculate", "read_doc"])
        t.eq("reader目录", [x["name"] for x in reader.request("tools/list")["tools"]], ["read_doc"])
        t.eq("private目录不能公共缓存", reader.request("tools/list")["cacheScope"], "private")
        t.eq("成功读取", reader.call_tool("read_doc",{"doc_id":"backup"})["data"]["doc_id"], "backup")
        t.eq("成功计算", operator.call_tool("calculate",{"op":"subtract","left":"30","right":"14"})["data"]["value"], "16")
        t.eq("Decimal计算", operator.call_tool("calculate",{"op":"add","left":"0.1","right":"0.2"})["data"]["value"], "0.3")
        t.eq("除零工具错误", operator.call_tool("calculate",{"op":"divide","left":"1","right":"0"})["error"], "division_by_zero")
        t.eq("参数校验业务错误", operator.call_tool("calculate",{"op":"add","left":1,"right":"2"})["error"], "invalid_arguments")
        before_exec = s.app.executions
        t.raises("reader不得直接调用计算", HTTPFailure, lambda: reader.call_tool("calculate",{"op":"add","left":"1","right":"2"}))
        t.eq("scope拒绝不执行", s.app.executions, before_exec)
        r = reader.raw_rpc(make_request("tools/call",{"name":"calculate","arguments":{"op":"add","left":"1","right":"2"}}, client_name="administrator"))
        t.eq("clientInfo不能提权", r["status"], 403)
        t.yes("缺scope挑战", 'scope="calc:use"' in r["headers"]["www-authenticate"])
        r = bob.raw_rpc(make_request("tools/call",{"name":"read_doc","arguments":{"doc_id":"alice_private"}}))
        t.eq("对象ACL不等于scope", r["status"], 403)
        t.eq("本人对象合法", reader.call_tool("read_doc",{"doc_id":"alice_private"})["ok"], True)
        t.eq("不存在文档是业务失败", reader.call_tool("read_doc",{"doc_id":"missing"})["error"], "not_found")
        for name, options in (("expired",{"ttl":-1}), ("audience",{"audience":"https://other.example/mcp"}),
                              ("issuer",{"issuer":"https://other.example"}), ("notyet",{"not_before":time.time()+3600})):
            token = s.app.tokens.provision("alice",["docs:read"],**options)
            rr = HttpClient(s.endpoint,token).raw_rpc(make_request("tools/list"))
            t.eq("真实HTTP拒绝"+name, rr["status"], 401)
        token = s.app.tokens.provision("alice",["docs:read"])
        authclient = HttpClient(s.endpoint,token)
        authclient.request("tools/list")
        s.app.tokens.revoke(token)
        t.raises("每次请求重新验证撤销", HTTPFailure, lambda: authclient.request("tools/list"))
        b = make_request("tools/call",{"name":"read_doc","arguments":{"doc_id":"backup"}})
        for header in ("MCP-Protocol-Version", "Mcp-Method", "Mcp-Name"):
            for bad in (None,"wrong"):
                rr = operator.raw_rpc(b,{header:bad})
                t.eq("镜像错误HTTP"+header+str(bad), rr["status"], 400)
                t.eq("镜像错误RPC"+header+str(bad), rr["body"]["error"]["code"], -32020)
        old = copy.deepcopy(b)
        old["params"]["_meta"][META+"protocolVersion"] = "2025-11-25"
        rr = operator.raw_rpc(old)
        t.eq("旧版本明确拒绝", rr["body"]["error"]["code"], -32022)
        rr = operator.raw_rpc(b,{"Origin":"https://untrusted.example"})
        t.eq("非法Origin403", rr["status"], 403)
        rr = operator.raw_rpc(b,{"Origin":s.app.origin})
        t.eq("同源仍需且有token", rr["status"], 200)
        rr = HttpClient(s.endpoint).raw_rpc(b,{"Origin":s.app.origin,"Mcp-Session-Id":"fake-admin"})
        t.eq("Origin和session不能替代token", rr["status"], 401)
        rr = operator.raw_rpc(b,{"Mcp-Session-Id":"ignored","Last-Event-ID":"ignored"})
        t.eq("旧元数据不建立会话", rr["status"], 200)
        t.yes("不回显session", "mcp-session-id" not in rr["headers"])
        t.eq("GETmcp405", operator.exchange("GET","/mcp",{})["status"], 405)
        t.eq("DELETEmcp405", operator.exchange("DELETE","/mcp",{})["status"], 405)
        t.eq("Host错误403", operator.raw_rpc(b,{"Host":"attacker.example"})["status"], 403)
        t.eq("Accept需两种响应", operator.raw_rpc(b,{"Accept":"application/json"})["status"], 406)
        t.eq("ContentType错误415", operator.raw_rpc(b,{"Content-Type":"text/plain"})["status"], 415)
        t.raises("未知方法是RPC失败", RPCFailure, lambda: operator.request("initialize"))
        t.raises("未知工具不是业务成功", RPCFailure, lambda: operator.request("tools/call",{"name":"shell","arguments":{}}))
        rr = operator.exchange("GET","/mcp?access_token=DO_NOT_LOG",{})
        t.eq("URL不允许携带token", rr["status"], 400)
        restricted = HttpClient(s.endpoint,s.tokens["operator"],allowed_tools={"read_doc"})
        t.raises("宿主拒绝高权限token动作", ContractError, lambda: restricted.call_tool("calculate",{"op":"add","left":"1","right":"2"}))
        t.eq("宿主拒绝在发包前", restricted.sent, 0)
        tiny = HttpClient(s.endpoint,s.tokens["reader"],budget=1)
        tiny.request("tools/list")
        t.raises("客户端请求预算", ContractError, lambda: tiny.request("tools/list"))
        audit = canonical(s.app.audit)
        t.yes("审计不含原始凭证", all(v not in audit for v in s.tokens.values()))
        t.yes("审计不含querytoken", "DO_NOT_LOG" not in audit)
        t.eq("缓存关闭", rr["headers"]["cache-control"], "no-store")
        t.yes("未开放浏览器CORS", "access-control-allow-origin" not in rr["headers"])
        # 重复身份头不是选一个使用，而是整体拒绝。
        dup = http.client.HTTPConnection("127.0.0.1",s.server.server_port,timeout=3)
        encoded = canonical(b).encode("utf-8")
        before_exec = s.app.executions
        dup.putrequest("POST","/mcp")
        for k,v in request_headers(b).items(): dup.putheader(k,v)
        dup.putheader("Authorization","Bearer "+s.tokens["reader"])
        dup.putheader("Authorization","Bearer "+s.tokens["operator"])
        dup.putheader("Content-Length",str(len(encoded)))
        dup.endheaders(encoded)
        response = dup.getresponse()
        t.eq("重复Authorization拒绝",response.status,400)
        response.read(); dup.close()
        t.eq("重复身份头不执行",s.app.executions,before_exec)

    with LocalService("sse") as s:
        c = HttpClient(s.endpoint,s.tokens["operator"])
        result = c.call_tool("read_doc",{"doc_id":"backup"})
        t.eq("真实SSE中文结果", result["data"]["text"], DOCS["backup"]["text"])
        t.eq("真实SSE媒体类型", c.wire[-1]["content_type"], "text/event-stream")
        t.eq("进度通知不替代final", c.wire[-1]["progress_notifications"], 1)
        t.eq("SSE业务失败不变成功", c.call_tool("calculate",{"op":"divide","left":"1","right":"0"})["ok"], False)
        t.eq("SSE服务器发现仍可JSON", c.request("server/discover")["supportedVersions"],[PROTOCOL])
    redirect = redirect_experiment()
    t.eq("拒绝重定向", redirect["blocked"], True)
    t.eq("不把token发送到Location", redirect["destination_requests"], 0)
    t.eq("重定向不自动重试", redirect["source_requests"], 1)
    t.eq("重定向实际建连次数", redirect["client_requests"], 1)
    return {"count":len(t.names), "passed":t.names,
            "tested":"standard_library_real_loopback_http_development_token_registry",
            "not_tested":["真实OAuth签发登录","TLS公网服务","官方SDK互操作","模型和GPU","生产安全认证"]}

### 六种有意错误
先确保正确实现通过检查，再临时替换某个函数，确认对应检查会失败，最后恢复原函数。依次删除 audience、scope、对象归属、镜像检查，并模拟 EOF 假成功与忽略 `isError`。

In [35]:
def mutation_checks() -> dict:
    entries = []
    def expect_rejected(fn, kinds):
        try:
            fn()
        except kinds:
            return
        raise AssertionError("应拒绝却放行")
    r = {"subject":"alice","scopes":["docs:read"],"issuer":ISSUER,"audience":"A",
         "expires_at":100,"not_before":0,"revoked":False}
    base_body = make_request("tools/call",{"name":"calculate","arguments":{"op":"add","left":"1","right":"2"}})
    bad_headers = {k.lower():v for k,v in request_headers(base_body).items()}
    bad_headers["mcp-name"] = "read_doc"
    originals = {"validate_grant": validate_grant, "validate_mirrors":validate_mirrors,
                 "require_scope":require_scope, "require_document_access":require_document_access,
                 "collect_sse":collect_sse, "validate_tool_result":validate_tool_result}
    cases = [
      ("遗漏audience检查","validate_grant",
       lambda rec,res,iss,now: originals["validate_grant"]({**rec,"audience":res},res,iss,now),
       lambda: expect_rejected(lambda: validate_grant(r,"B",ISSUER,1),(GrantRejected,))),
      ("跳过scope检查","require_scope",lambda p,n:None,
       lambda: expect_rejected(lambda: require_scope(Principal("alice",frozenset({"docs:read"}),ISSUER,"A"),"calculate"),(PermissionError,))),
      ("跳过对象归属","require_document_access",lambda p,d:None,
       lambda: expect_rejected(lambda: require_document_access(Principal("bob",frozenset({"docs:read"}),ISSUER,"A"),"alice_private"),(PermissionError,))),
      ("镜像头仅用于显示","validate_mirrors",lambda b,h:None,
       lambda: expect_rejected(lambda: validate_mirrors(base_body,bad_headers),(ContractError,))),
      ("把EOF当成功","collect_sse",lambda messages,rid,progress_token=None:(rpc_result(rid,{}),0),
       lambda: expect_rejected(lambda: collect_sse([],1),(ContractError,))),
      ("忽略isError","validate_tool_result",lambda result:{"ok":True,"data":{},"error":None},
       lambda: _check_failed_tool()),
    ]
    for name, key, mutant, probe in cases:
        probe()  # 先验证未改动时检查成立。
        globals()[key] = mutant
        detected = False
        try:
            probe()
        except AssertionError:
            detected = True
        finally:
            globals()[key] = originals[key]
        if not detected:
            raise AssertionError("故障未被检测："+name)
        entries.append({"mutation":name,"detected":True})
    return {"count":len(entries),"results":entries,"scope":"函数级有意替换，不是渗透测试"}


def _check_failed_tool():
    result = validate_tool_result({"resultType":"complete",**tool_result(error="division_by_zero")})
    if result["ok"] is not False:
        raise AssertionError("业务失败被当作成功")

In [36]:
checks = run_checks()
print("固定检查通过：", checks["count"])
print("验证范围：", checks["tested"])
print("未测试：", checks["not_tested"])

固定检查通过： 110
验证范围： standard_library_real_loopback_http_development_token_registry
未测试： ['真实OAuth签发登录', 'TLS公网服务', '官方SDK互操作', '模型和GPU', '生产安全认证']


In [37]:
mutations = mutation_checks()
for result in mutations["results"]:
    print(result["mutation"], "→ 检测到：", result["detected"])

遗漏audience检查 → 检测到： True
跳过scope检查 → 检测到： True
跳过对象归属 → 检测到： True
镜像头仅用于显示 → 检测到： True
把EOF当成功 → 检测到： True
忽略isError → 检测到： True


## 15．保存本次结果

保存的报告没有开发 token。HTTP trace 只有固定路径、状态、请求 ID、内容摘要和通知数量；这既减少凭证泄漏，也不是完整防篡改审计。

如果把读取工具换成私人资料，仍需审查哪些返回内容可以保存或分享。

In [38]:
def write_json(path: Path, value: Any):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+"\n",encoding="utf-8")


def output_dir(root: str | Path) -> Path:
    path = Path(root) / (datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_")+secrets.token_hex(3))
    path.mkdir(parents=True,exist_ok=False)
    return path


def environment() -> dict:
    return {"python":sys.version.split()[0],"platform":platform.platform(),"protocol":PROTOCOL,
            "dependencies":"Python标准库","network":"127.0.0.1 only","oauth":"development registry only"}


def serve(port: int, mode: str):
    with LocalService(mode,port) as s:
        # 唯一会打印真实开发 token 的界面：启动者的本机终端，不写报告、不返回HTTP。
        print("仅本机教学服务；这些临时凭证会随进程退出失效。不要复制到模型提示或公开日志。",flush=True)
        print("endpoint:",s.endpoint,flush=True)
        print("reader_token:",s.tokens["reader"],flush=True)
        print("operator_token:",s.tokens["operator"],flush=True)
        print("reader=alice/docs:read；operator=alice/docs:read+calc:use；不是生产管理员。",flush=True)
        print("Ctrl+C 关闭。服务端没有 /token、/authorize 或刷新端点。",flush=True)
        try:
            while True:
                time.sleep(0.2)
        except KeyboardInterrupt:
            pass

In [39]:
NOTEBOOK_OUTPUT_ROOT = "lesson_52_notebook_outputs"
out = output_dir(NOTEBOOK_OUTPUT_ROOT)
write_json(out / "checks.json", checks)
write_json(out / "experiment.json", experiment)
write_json(out / "mutations.json", mutations)
write_json(out / "redirect.json", redirect_report)
write_json(out / "manifest.json", {
    "version": VERSION, "environment": environment(), "sources": SOURCES,
    "status": "completed", "credential_values_saved": False,
})
print("报告已保存：", out.resolve())

报告已保存： /mnt/data/_lesson52_work/notebook_run/lesson_52_notebook_outputs/run_20261102T125547_889614


## 16．用两个终端亲手操作

**终端 A：**

```bash
python lesson_52_mcp_http_identity_and_authorization.py --serve --port 8768 --response-mode sse
```

它会打印 `reader_token` 与 `operator_token`。它们是每次启动随机生成、默认900秒有效的开发凭证，仅在该进程中存在。不要发给模型、上传仓库或贴进共享日志。

**终端 B，先选择 reader：**

Bash：
```bash
export LESSON52_ACCESS_TOKEN='终端A的reader_token'
```

PowerShell：
```powershell
$env:LESSON52_ACCESS_TOKEN = '终端A的reader_token'
```

```bash
python lesson_52_mcp_http_identity_and_authorization.py --client http://127.0.0.1:8768/mcp --action list
python lesson_52_mcp_http_identity_and_authorization.py --client http://127.0.0.1:8768/mcp --action read --doc backup
python lesson_52_mcp_http_identity_and_authorization.py --client http://127.0.0.1:8768/mcp --action calculate
```

前两条成功，第三条返回 HTTP 403，退出码 2。把环境变量切换为 **operator_token** 后，最后一条得到 `16`。

operator 在本课只是 Alice 的 `docs:read + calc:use` 凭证，不是拥有所有权限的管理员。即使有它，读取 `bob_private` 仍会被拒绝。

仅观察公开的授权发现，不需要设置凭证：
```bash
python lesson_52_mcp_http_identity_and_authorization.py --client http://127.0.0.1:8768/mcp --action discover-auth
```

运行 `--serve` 之前不要把其他课程的 token 复制过来，它们不是本课服务签发/预配的凭证。到期或重启后重新取得本次开发凭证，而不是无限重试。

Ctrl+C 关闭服务。HTTP 200 携带工具业务失败时，CLI 用退出码 3；HTTP/RPC/传输拒绝用 2。

## 17．验收与边界

请能解释：

1. 为什么 reader 可以列出工具，却不能调用计算器？目录隐藏是否足够？
2. 为什么同一 subject、同一 scope 的 token，发给另一个 audience 仍然无效？
3. 为什么设置 `clientInfo.name`、`Origin` 或旧 session ID 都不等于身份认证？
4. 为什么允许工具还要检查具体文档归属？
5. 为什么 HTTP 200、SSE 最终消息和工具业务成功是不同判断？
6. 为什么收到 metadata 后不能直接把当前 Bearer token 发给其中任意 URL？

**本课没有实现：**正式 OAuth/OIDC 登录、PKCE、JWT/JWKS 验签、refresh、TLS、公共网络服务、浏览器 CORS、分布式 token 撤销、写操作审批、MRTR、订阅、完整JSON Schema、MCP兼容认证或生产压测。没有调用模型，不能据此报告 Agent 规划/抗注入能力。

标准库 `http.server` 仅适用于本机教学；不要通过改成 `0.0.0.0` 就把本课作为生产服务暴露。来源：[8]。

下一课：**第53课，OAuth 2.1、PKCE 与授权流程**。继续区分可信客户端配置、用户授权、令牌获取与资源访问，不把开发凭证表当作完整身份基础设施。

## 原始资料

[1] [MCP 2026-07-28 Streamable HTTP](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/streamable-http)：请求、SSE、镜像头与旧会话的差异。

[2] [MCP Discovery](https://modelcontextprotocol.io/specification/2026-07-28/server/discover)：服务能力与自报身份的限制。

[3] [MCP Authorization](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization)：OAuth角色、受众绑定、凭证位置、401/403。

[4] [MCP Tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools)：目录、权限变化、结构化结果与工具错误。

[5] [MCP Security Best Practices](https://modelcontextprotocol.io/docs/2026-07-28/tutorials/security/security_best_practices)：SSRF、token passthrough、发行方混淆等风险。

[6] [Authorization Server Discovery](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/authorization-server-discovery)：授权服务发现，不等于 `server/discover`。

[7] [WHATWG Server-Sent Events](https://html.spec.whatwg.org/multipage/server-sent-events.html)：UTF-8 与事件边界。

[8] [Python http.server](https://docs.python.org/3/library/http.server.html)：标准库服务的使用边界。

[9] [官方 Python SDK 的授权页](https://py.sdk.modelcontextprotocol.io/run/authorization/)：生产服务中替换开发 token verifier 的方向；本课未执行SDK互操作。

本课固定规范版本，而不是承诺以后所有版本行为不变。